# Project Solution: Lazy Polygon Properties and Lazy Polygon Iterable

This notebook contains a refactored solution for the previous `Polygon` / `Polygons` project.

## Goals covered

1. `Polygon` calculated properties are lazy and cached after first access.
2. `Polygons` is an iterable that creates polygon objects lazily.
3. `Polygons` no longer stores a pre-built list of polygons.
4. A separate iterator type is implemented.
5. Tests verify correctness, immutability behavior, lazy caching, and iterator independence.

In [1]:
import math
import numbers
from functools import total_ordering
from itertools import islice

## Lazy property descriptor

`LazyProperty` is a small reusable descriptor.

Why not use `functools.cached_property`?

- This solution works in older Python versions where `cached_property` may not exist.
- It also works cleanly with `__slots__`.
- The cached values live in a dedicated `_lazy_cache` dictionary.

In [2]:
class LazyProperty:
    """Descriptor that computes a value once per instance and caches it.

    The owner class must provide a `_lazy_cache` dictionary.
    """

    __slots__ = ("func", "name")

    def __init__(self, func):
        self.func = func
        self.name = func.__name__

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self

        cache = instance._lazy_cache

        if self.name not in cache:
            cache[self.name] = self.func(instance)

        return cache[self.name]

## Polygon

Design notes:

- Validates inputs eagerly.
- Uses `__slots__` to avoid accidental dynamic attributes.
- Implements frozen-like behavior after initialization.
- Uses lazy cached properties for all public derived values.
- Supports equality and total ordering.
- Ordering is by `(number of vertices, circumradius)`, giving a complete and consistent ordering.

In [3]:
@total_ordering
class Polygon:
    """Regular polygon defined by number of vertices and circumradius.

    Parameters
    ----------
    n:
        Number of vertices / edges. Must be an integer >= 3.
    R:
        Circumradius. Must be a positive finite real number.
    """

    __slots__ = ("_n", "_R", "_lazy_cache", "_initialized")

    def __init__(self, n, R):
        self._validate_n(n)
        self._validate_R(R)

        object.__setattr__(self, "_n", n)
        object.__setattr__(self, "_R", R)
        object.__setattr__(self, "_lazy_cache", {})
        object.__setattr__(self, "_initialized", True)

    @staticmethod
    def _validate_n(n):
        if isinstance(n, bool) or not isinstance(n, int):
            raise TypeError("n must be an integer.")
        if n < 3:
            raise ValueError("Polygon must have at least 3 vertices.")

    @staticmethod
    def _validate_R(R):
        if isinstance(R, bool) or not isinstance(R, numbers.Real):
            raise TypeError("R must be a real number.")
        if not math.isfinite(R):
            raise ValueError("R must be finite.")
        if R <= 0:
            raise ValueError("R must be positive.")

    def __setattr__(self, name, value):
        if getattr(self, "_initialized", False):
            raise AttributeError(f"{self.__class__.__name__} instances are immutable.")
        object.__setattr__(self, name, value)

    def __delattr__(self, name):
        raise AttributeError(f"{self.__class__.__name__} instances are immutable.")

    def __repr__(self):
        return f"Polygon(n={self._n}, R={self._R})"

    @LazyProperty
    def count_vertices(self):
        return self._n

    @LazyProperty
    def count_edges(self):
        return self._n

    @LazyProperty
    def circumradius(self):
        return self._R

    @LazyProperty
    def interior_angle(self):
        return (self._n - 2) * 180 / self._n

    @LazyProperty
    def side_length(self):
        return 2 * self._R * math.sin(math.pi / self._n)

    @LazyProperty
    def apothem(self):
        return self._R * math.cos(math.pi / self._n)

    @LazyProperty
    def area(self):
        return self._n / 2 * self.side_length * self.apothem

    @LazyProperty
    def perimeter(self):
        return self._n * self.side_length

    @LazyProperty
    def efficiency(self):
        """Area-to-perimeter ratio."""
        return self.area / self.perimeter

    def __eq__(self, other):
        if not isinstance(other, Polygon):
            return NotImplemented

        return (
            self.count_edges == other.count_edges
            and self.circumradius == other.circumradius
        )

    def __lt__(self, other):
        if not isinstance(other, Polygon):
            return NotImplemented

        return (
            self.count_vertices,
            self.circumradius,
        ) < (
            other.count_vertices,
            other.circumradius,
        )

## Polygons iterator

The iterator owns its iteration state.

Each call to `next()` creates exactly one `Polygon` object and then advances the iterator.

In [4]:
class PolygonsIterator:
    """Iterator for regular polygons from 3 vertices through m vertices."""

    __slots__ = ("_current_n", "_max_n", "_R")

    def __init__(self, max_n, R):
        self._current_n = 3
        self._max_n = max_n
        self._R = R

    def __iter__(self):
        return self

    def __next__(self):
        if self._current_n > self._max_n:
            raise StopIteration

        polygon = Polygon(self._current_n, self._R)
        self._current_n += 1
        return polygon

## Polygons iterable

`Polygons` is now an iterable, not a sequence backed by a list.

Important points:

- No `_polygons` list is created.
- `__iter__` returns a new independent iterator every time.
- `max_efficiency_polygon` is also lazy and cached.
- `__len__` is still useful and cheap, so it is kept.

In [5]:
class Polygons:
    """Iterable of regular polygons from 3 vertices through m vertices.

    Parameters
    ----------
    m:
        Maximum number of vertices. Must be an integer >= 3.
    R:
        Circumradius used for every polygon. Must be a positive finite real number.
    """

    __slots__ = ("_m", "_R", "_lazy_cache", "_initialized")

    def __init__(self, m, R):
        Polygon._validate_n(m)
        Polygon._validate_R(R)

        object.__setattr__(self, "_m", m)
        object.__setattr__(self, "_R", R)
        object.__setattr__(self, "_lazy_cache", {})
        object.__setattr__(self, "_initialized", True)

    def __setattr__(self, name, value):
        if getattr(self, "_initialized", False):
            raise AttributeError(f"{self.__class__.__name__} instances are immutable.")
        object.__setattr__(self, name, value)

    def __delattr__(self, name):
        raise AttributeError(f"{self.__class__.__name__} instances are immutable.")

    def __repr__(self):
        return f"Polygons(m={self._m}, R={self._R})"

    def __len__(self):
        return self._m - 2

    def __iter__(self):
        return PolygonsIterator(self._m, self._R)

    @LazyProperty
    def max_efficiency_polygon(self):
        return max(self, key=lambda polygon: polygon.efficiency)

## Tests for `Polygon`

These tests include the original checks plus validation, immutability, and lazy caching checks.

In [6]:
def test_polygon():
    abs_tol = 0.001
    rel_tol = 0.001

    invalid_inputs = [
        (2, 10, ValueError),
        (3.5, 10, TypeError),
        (True, 10, TypeError),
        (3, 0, ValueError),
        (3, -1, ValueError),
        (3, float("inf"), ValueError),
        (3, "10", TypeError),
    ]

    for n, R, expected_exception in invalid_inputs:
        try:
            Polygon(n, R)
            assert False, f"Creating Polygon({n!r}, {R!r}) should have failed."
        except expected_exception:
            pass

    n = 3
    R = 1
    p = Polygon(n, R)

    assert str(p) == "Polygon(n=3, R=1)", f"actual: {str(p)}"
    assert p.count_vertices == n, f"actual: {p.count_vertices}, expected: {n}"
    assert p.count_edges == n, f"actual: {p.count_edges}, expected: {n}"
    assert p.circumradius == R, f"actual: {p.circumradius}, expected: {R}"
    assert p.interior_angle == 60, f"actual: {p.interior_angle}, expected: 60"

    n = 4
    R = 1
    p = Polygon(n, R)

    assert p.interior_angle == 90, f"actual: {p.interior_angle}, expected: 90"

    assert math.isclose(
        p.area,
        2,
        rel_tol=rel_tol,
        abs_tol=abs_tol,
    ), f"actual: {p.area}, expected: 2.0"

    assert math.isclose(
        p.side_length,
        math.sqrt(2),
        rel_tol=rel_tol,
        abs_tol=abs_tol,
    ), f"actual: {p.side_length}, expected: {math.sqrt(2)}"

    assert math.isclose(
        p.perimeter,
        4 * math.sqrt(2),
        rel_tol=rel_tol,
        abs_tol=abs_tol,
    ), f"actual: {p.perimeter}, expected: {4 * math.sqrt(2)}"

    assert math.isclose(
        p.apothem,
        0.707,
        rel_tol=rel_tol,
        abs_tol=abs_tol,
    ), f"actual: {p.apothem}, expected: 0.707"

    p = Polygon(6, 2)

    assert math.isclose(p.side_length, 2, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.apothem, 1.73205, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.area, 10.3923, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.perimeter, 12, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.interior_angle, 120, rel_tol=rel_tol, abs_tol=abs_tol)

    p = Polygon(12, 3)

    assert math.isclose(p.side_length, 1.55291, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.apothem, 2.89778, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.area, 27, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.perimeter, 18.635, rel_tol=rel_tol, abs_tol=abs_tol)
    assert math.isclose(p.interior_angle, 150, rel_tol=rel_tol, abs_tol=abs_tol)

    p1 = Polygon(3, 10)
    p2 = Polygon(10, 10)
    p3 = Polygon(15, 10)
    p4 = Polygon(15, 100)
    p5 = Polygon(15, 100)

    assert p2 > p1
    assert p2 < p3
    assert p3 != p4
    assert p1 != p4
    assert p4 == p5

    # Immutability checks
    p = Polygon(5, 10)

    try:
        p._n = 99
        assert False, "Expected AttributeError when modifying _n."
    except AttributeError:
        pass

    try:
        p.new_attribute = "not allowed"
        assert False, "Expected AttributeError when adding a new attribute."
    except AttributeError:
        pass

    try:
        del p._R
        assert False, "Expected AttributeError when deleting _R."
    except AttributeError:
        pass

    # Lazy caching checks
    p = Polygon(6, 2)
    assert p._lazy_cache == {}

    first_side_length = p.side_length
    assert "side_length" in p._lazy_cache

    second_side_length = p.side_length
    assert first_side_length == second_side_length
    assert len([key for key in p._lazy_cache if key == "side_length"]) == 1

    p = Polygon(6, 2)
    _ = p.area

    assert "area" in p._lazy_cache
    assert "side_length" in p._lazy_cache
    assert "apothem" in p._lazy_cache

    print("Polygon tests passed.")

In [7]:
test_polygon()

Polygon tests passed.


## Tests for `Polygons`

These tests verify that `Polygons` behaves as an iterable and does not rely on an internal list of polygons.

In [8]:
def test_polygons():
    invalid_inputs = [
        (2, 10, ValueError),
        (3.5, 10, TypeError),
        (True, 10, TypeError),
        (3, 0, ValueError),
        (3, -1, ValueError),
        (3, float("inf"), ValueError),
        (3, "10", TypeError),
    ]

    for m, R, expected_exception in invalid_inputs:
        try:
            Polygons(m, R)
            assert False, f"Creating Polygons({m!r}, {R!r}) should have failed."
        except expected_exception:
            pass

    polygons = Polygons(6, 10)

    assert repr(polygons) == "Polygons(m=6, R=10)"
    assert len(polygons) == 4

    # No pre-built list should exist.
    assert not hasattr(polygons, "_polygons")

    # Polygons is an iterable, not its own iterator.
    iterator = iter(polygons)

    assert iterator is not polygons
    assert iter(iterator) is iterator

    first = next(iterator)
    second = next(iterator)

    assert first == Polygon(3, 10)
    assert second == Polygon(4, 10)

    remaining = list(iterator)

    assert remaining == [
        Polygon(5, 10),
        Polygon(6, 10),
    ]

    try:
        next(iterator)
        assert False, "Expected StopIteration."
    except StopIteration:
        pass

    # Each call to iter(polygons) returns an independent iterator.
    iterator_1 = iter(polygons)
    iterator_2 = iter(polygons)

    assert next(iterator_1) == Polygon(3, 10)
    assert next(iterator_1) == Polygon(4, 10)
    assert next(iterator_2) == Polygon(3, 10)

    # The iterable can be consumed repeatedly.
    assert list(polygons) == [
        Polygon(3, 10),
        Polygon(4, 10),
        Polygon(5, 10),
        Polygon(6, 10),
    ]

    assert list(polygons) == [
        Polygon(3, 10),
        Polygon(4, 10),
        Polygon(5, 10),
        Polygon(6, 10),
    ]

    # Lazy iteration: this only creates the first two Polygon objects.
    first_two = list(islice(Polygons(100_000, 1), 2))

    assert first_two == [
        Polygon(3, 1),
        Polygon(4, 1),
    ]

    # max_efficiency_polygon is lazy and cached.
    polygons = Polygons(25, 10)

    assert polygons._lazy_cache == {}

    best = polygons.max_efficiency_polygon

    assert best == Polygon(25, 10)
    assert "max_efficiency_polygon" in polygons._lazy_cache
    assert polygons.max_efficiency_polygon is best

    # Immutability checks
    try:
        polygons._m = 99
        assert False, "Expected AttributeError when modifying _m."
    except AttributeError:
        pass

    try:
        polygons.new_attribute = "not allowed"
        assert False, "Expected AttributeError when adding a new attribute."
    except AttributeError:
        pass

    print("Polygons tests passed.")

In [9]:
test_polygons()

Polygons tests passed.


## Small usage demo

In [10]:
polygons = Polygons(8, 5)

print(polygons)
print(f"Length: {len(polygons)}")
print("Generated lazily:")

for polygon in polygons:
    print(
        polygon,
        "area=",
        round(polygon.area, 4),
        "perimeter=",
        round(polygon.perimeter, 4),
        "efficiency=",
        round(polygon.efficiency, 4),
    )

print("Best efficiency:", polygons.max_efficiency_polygon)

Polygons(m=8, R=5)
Length: 6
Generated lazily:
Polygon(n=3, R=5) area= 32.476 perimeter= 25.9808 efficiency= 1.25
Polygon(n=4, R=5) area= 50.0 perimeter= 28.2843 efficiency= 1.7678
Polygon(n=5, R=5) area= 59.441 perimeter= 29.3893 efficiency= 2.0225
Polygon(n=6, R=5) area= 64.9519 perimeter= 30.0 efficiency= 2.1651
Polygon(n=7, R=5) area= 68.4103 perimeter= 30.3719 efficiency= 2.2524
Polygon(n=8, R=5) area= 70.7107 perimeter= 30.6147 efficiency= 2.3097
Best efficiency: Polygon(n=8, R=5)


## Key takeaways

- `Polygon` values such as `area`, `perimeter`, `side_length`, and `apothem` are computed on first access only.
- `Polygons` does not store a list of `Polygon` instances.
- Iteration over `Polygons` creates each `Polygon` only when requested.
- Repeated iterations are safe because `Polygons.__iter__` returns a fresh iterator each time.
- `max_efficiency_polygon` is cached after first access.